# 90 · Entrenar un modelo propio

**Complementario.** Fuera del camino obligatorio.

## Propósito

En el cuaderno `05` corregiste el NER con reglas. Las reglas tienen un techo:
solo encuentran lo que ya está en tu lista.

Entrenar un modelo es la alternativa. Acá vamos a hacerlo de punta a punta,
**y va a salir aparentemente perfecto**. Entender por qué esa perfección es una
mala noticia es todo el contenido de este cuaderno.

## Qué consume

- `datos/corpus.spacy`
- `pipeline_propio/` y `componentes.py`

## Advertencia

Esto no entra en las cuatro horas de clase. Requiere paciencia y el
entrenamiento tarda alrededor de un minuto.

### ⚙ Configuración

## Celda de reanudación

In [ ]:
# spacy: la librería de análisis lingüístico.
import spacy

# random: para mezclar los ejemplos de forma reproducible.
import random

# time: para medir cuánto tarda el entrenamiento.
import time

# DocBin: el contenedor de documentos procesados.
from spacy.tokens import DocBin

# Example: el formato en que spaCy espera los ejemplos de entrenamiento.
from spacy.training import Example

# minibatch: para partir los ejemplos en lotes.
from spacy.util import minibatch

# Este cuaderno vive en complementarios/, pero el corpus, la herramienta y
# componentes.py están una carpeta más arriba. Le decimos a Python dónde buscar.
import sys
sys.path.append("..")

# Nuestros componentes propios.
import componentes

nlp = spacy.load("../pipeline_propio")

banco = DocBin().from_disk("../datos/corpus.spacy")
documentos = list(banco.get_docs(nlp.vocab))

cantidad_documentos = len(documentos)
print("documentos:", f"{cantidad_documentos:,}")

assert cantidad_documentos > 0, (
    "No se recuperó ningún documento. Este cuaderno necesita ../datos/corpus.spacy, "
    "que produce el 06."
)

## 1. El problema de los datos anotados

Para entrenar un modelo hacen falta ejemplos: textos donde alguien marcó a mano
dónde están las entidades y de qué tipo son.

Anotar a mano es caro. Un atajo muy usado es la **supervisión débil**: generar
las anotaciones con reglas, y entrenar el modelo con eso.

Nosotros ya tenemos las reglas del cuaderno `05`, así que el atajo está servido.
Vamos a tomarlo, y después vamos a ver qué precio tiene.

In [ ]:
ejemplos = []

for documento in documentos:
    entidades = []
    for entidad in documento.ents:
        if entidad.label_ != "PERSONAJE":
            continue
        entidades.append((entidad.start_char, entidad.end_char, "PERSONAJE"))
    if entidades:
        ejemplos.append((documento.text, {"entities": entidades}))

print("párrafos con al menos un PERSONAJE:", len(ejemplos))
print()
texto_ejemplo, anotacion_ejemplo = ejemplos[0]
print("un ejemplo:")
print("  texto     :", texto_ejemplo[:80], "...")
print("  entidades :", anotacion_ejemplo["entities"])

## 2. Partir en entrenamiento y prueba

Regla que no se negocia: **el modelo nunca puede ver, mientras aprende, los
datos con los que después lo vas a evaluar.**

In [ ]:
random.seed(0)
random.shuffle(ejemplos)

corte = int(len(ejemplos) * 0.8)
entrenamiento = ejemplos[:corte]
prueba = ejemplos[corte:]

print("entrenamiento:", len(entrenamiento))
print("prueba       :", len(prueba))

## 3. Entrenar

`spacy.blank("es")` crea un pipeline vacío con el idioma español: trae el
tokenizador y nada más. Le agregamos un `ner` sin entrenar y le decimos qué
etiquetas va a tener que aprender.

In [ ]:
modelo = spacy.blank("es")

reconocedor = modelo.add_pipe("ner")
reconocedor.add_label("PERSONAJE")

print("pipeline del modelo nuevo:", modelo.pipe_names)

In [ ]:
inicio = time.time()

# initialize() prepara los pesos iniciales y devuelve el optimizador.
optimizador = modelo.initialize()

for epoca in range(6):
    random.shuffle(entrenamiento)
    perdidas = {}
    for lote in minibatch(entrenamiento, size=32):
        ejemplos_del_lote = []
        for texto, anotacion in lote:
            documento = modelo.make_doc(texto)
            ejemplos_del_lote.append(Example.from_dict(documento, anotacion))
        modelo.update(ejemplos_del_lote, sgd=optimizador, losses=perdidas)
    print(f"época {epoca + 1}: pérdida {perdidas['ner']:.1f}")

print()
duracion = time.time() - inicio
print(f"tardó {duracion:.0f} segundos")

La pérdida baja a cero en cuatro épocas. Eso ya debería llamarte la atención:
un problema real no se resuelve tan rápido.

## 4. Evaluar

In [ ]:
ejemplos_de_prueba = []
for texto, anotacion in prueba:
    documento = modelo.make_doc(texto)
    ejemplos_de_prueba.append(Example.from_dict(documento, anotacion))

resultado = modelo.evaluate(ejemplos_de_prueba)

print("precisión :", round(resultado["ents_p"], 3))
print("cobertura :", round(resultado["ents_r"], 3))
print("F         :", round(resultado["ents_f"], 3))

## 5. Por qué ese resultado es una mala noticia

Precisión 1.0, cobertura 1.0, F 1.0. **Perfecto.**

Si publicaras eso sin pensar, estarías diciendo que entrenaste un reconocedor de
personajes impecable. Vamos a probar si es cierto.

In [ ]:
print("--- nombres que SÍ están en personajes.txt ---")
doc_conocidos = modelo("Chiri fue con Nina a la casa de Mónica.")
for entidad in doc_conocidos.ents:
    print(" ", entidad.text)

print()
print("--- nombres que NO están en la lista ---")
frases = [
    "Rodolfo salió con Estela y Ramiro.",
    "Ayer vino Jorgito a visitar a Susana.",
    "Federico y Alejandro se fueron temprano.",
]
for frase in frases:
    documento = modelo(frase)
    encontrados = []
    for entidad in documento.ents:
        encontrados.append(entidad.text)
    print(" ", frase, "->", encontrados)

**Cero.** El modelo no reconoce ningún nombre que no estuviera en la lista.

Lo que aprendió no fue "cómo se ve un personaje en este corpus". Aprendió
**la lista de `personajes.txt`**, que es exactamente lo que le dimos.

Y por eso la evaluación dio perfecta: el conjunto de prueba también estaba
anotado con la misma regla. Le preguntamos al modelo si había aprendido la regla
con la que lo entrenamos. Obviamente sí.

> ### La lección
>
> **Un F de 1.0 casi nunca significa que tu modelo es excelente. Significa que
> tu evaluación está mal planteada.**
>
> Acá el circuito era circular: reglas → anotaciones → modelo → evaluación contra
> las mismas reglas. En ningún punto entró información sobre el fenómeno que
> supuestamente estábamos modelando.
>
> Es un error frecuente y difícil de ver desde adentro, porque todos los números
> dan bien.

## 6. Qué haría falta para hacerlo bien

**Anotar a mano.** Alguien —vos— tiene que leer párrafos y marcar los personajes,
incluidos los que no están en ninguna lista. Ahí sí el modelo tendría algo que
generalizar.

**Evaluar contra anotación humana**, no contra la regla que generó el
entrenamiento.

**Medir la concordancia entre anotadores.** Si dos personas anotan el mismo
material y no coinciden, el problema no es el modelo: es que la categoría no
está bien definida. Y esa es una discusión conceptual, no técnica.

**Usar la supervisión débil como punto de partida y no como verdad.** Generás
anotaciones con reglas, las corregís a mano, y entrenás con las corregidas. Es
el uso legítimo del atajo.

Ahí está el volumen de trabajo real de cualquier proyecto de este tipo: **casi
todo es anotación, no modelado.**

## 7. Ejercicio

1. Anotá a mano 30 párrafos del corpus, marcando **todos** los personajes que
   veas, estén o no en `personajes.txt`. Evaluá el modelo entrenado contra esa
   anotación. ¿Cuánto baja el F?
2. Entrená otra vez agregando esos 30 párrafos al conjunto de entrenamiento.
   ¿Empieza a reconocer nombres nuevos?
3. Pedile a un compañero que anote los mismos 30 párrafos y comparen. ¿En cuántos
   coinciden? Las diferencias, ¿son errores o son desacuerdos sobre qué cuenta
   como personaje?

In [ ]:
# TU CÓDIGO ACÁ

## Guardar el modelo

Aunque no sirva para mucho, se guarda igual que cualquier pipeline.

In [ ]:
modelo.to_disk("ner_personajes")

recargado = spacy.load("ner_personajes")
documento = recargado("Chiri y Nina fueron a ver a Mirta Bertotti.")

print("entidades:")
for entidad in documento.ents:
    print("  ", entidad.text, "->", entidad.label_)